## **Notebook 12 — RAG + Long/Short Term Memory (End to End)**
Integrates the full production RAG pipeline with a persistent memory system.

- Document: data/docs/md_al_amin_resume.pdf
- STM: LangGraph PostgresSaver checkpointer — per-thread conversation history, summarised at 6 messages
- LTM: LangGraph PostgresStore — persistent user profile facts across all threads
- The LLM answers from the resume AND from what it remembers about the user across sessions.

# ***PART 1 — Imports & Model Loading***

## **Cell 1 — install + imports**
- **Why:** All libraries needed for the full pipeline in one place.
- **langgraph-checkpoint-postgres:** provides PostgresSaver (STM) and PostgresStore (LTM). These are official LangGraph persistence backends — they store conversation checkpoints and memory namespaces directly in your existing PostgreSQL database. No separate vector database needed for memory.


In [1]:
# Install if needed
# !pip install langgraph-checkpoint-postgres psycopg[binary] psycopg[pool]
# !pip install FlagEmbedding bm25s docling langchain-openai python-dotenv

import os, json, re, uuid, operator
from pathlib import Path
from typing import TypedDict, Annotated, Optional
from dataclasses import dataclass, field as dc_field
from collections import Counter
from datetime import datetime
from dotenv import load_dotenv

import numpy as np
import psycopg2
from pgvector.psycopg2 import register_vector
import bm25s

from docling.document_converter import DocumentConverter
from FlagEmbedding import BGEM3FlagModel, FlagReranker
from langchain_openai import ChatOpenAI
from langchain_core.messages import (
    SystemMessage, HumanMessage, AIMessage, RemoveMessage
)

# LangGraph core
from langgraph.graph import StateGraph, END
from langgraph.checkpoint.postgres import PostgresSaver
from langgraph.store.postgres import PostgresStore

load_dotenv()

DATABASE_SYNC = os.getenv("DATABASE_URL_SYNC")
OPENAI_KEY    = os.getenv("OPENAI_API_KEY")
INDEX_DIR     = Path("../data/bm25_index_resume")
INDEX_DIR.mkdir(parents=True, exist_ok=True)

print("Imports OK")
print(f"DB  : {DATABASE_SYNC[:10]}...")
print(f"Key : {OPENAI_KEY[:15]}...")

Imports OK
DB  : postgresql...
Key : sk-proj-Pm8wgjk...


## **Cell 2 — load AI models**
- **Why load all models here:** BGE-M3 (~2.3GB) and the reranker (~1.1GB) take 20-30s to load. Loading once at the top means every subsequent cell is instant.
- **GPT-4o-mini for everything:** We use gpt-4o-mini for HyDE generation, LTM fact extraction, STM summarisation, and final answer generation. It is fast, cheap, and accurate enough for all four tasks.

In [2]:
MODEL_DIR = Path("../models")

print("Loading BGE-M3 embedder...")
embed_model = BGEM3FlagModel(
    "BAAI/bge-m3", use_fp16=False,
    cache_dir=str(MODEL_DIR / "bge-m3")
)
print("  BGE-M3 ready")

print("Loading BGE reranker...")
rerank_model = FlagReranker(
    "BAAI/bge-reranker-v2-m3", use_fp16=False,
    cache_dir=str(MODEL_DIR / "bge-reranker")
)
print("  Reranker ready")

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.1)
print("  LLM ready (gpt-4o-mini)")

print("\nAll models loaded OK")

Loading BGE-M3 embedder...


Fetching 30 files:   0%|          | 0/30 [00:00<?, ?it/s]

  BGE-M3 ready
Loading BGE reranker...
  Reranker ready
  LLM ready (gpt-4o-mini)

All models loaded OK


## ***Cell 3 — fresh DB tables for this notebook**
- **Why fresh tables:** Isolated from previous notebooks so you can re-run from scratch without corrupting earlier experiments.
chunks_resume + documents_resume: Same schema as before but with suffix to avoid collisions.
- **pgvector HNSW index:** Created immediately after table creation since we have few chunks (resume is small). In production with large corpora, create the HNSW index AFTER bulk loading all data for performance.

In [3]:
conn_str = DATABASE_SYNC.replace("postgresql+psycopg2://", "postgresql://")
conn     = psycopg2.connect(conn_str)
register_vector(conn)
cur      = conn.cursor()

# Drop and recreate for clean start
cur.execute("DROP TABLE IF EXISTS chunks_resume   CASCADE;")
cur.execute("DROP TABLE IF EXISTS documents_resume CASCADE;")

cur.execute("""
    CREATE TABLE documents_resume (
        id         UUID PRIMARY KEY DEFAULT gen_random_uuid(),
        filename   TEXT NOT NULL,
        status     TEXT DEFAULT 'ready',
        created_at TIMESTAMPTZ DEFAULT now()
    );
""")

cur.execute("""
    CREATE TABLE chunks_resume (
        id             UUID PRIMARY KEY DEFAULT gen_random_uuid(),
        document_id    UUID REFERENCES documents_resume(id) ON DELETE CASCADE,
        content        TEXT NOT NULL,
        parent_content TEXT,
        embedding      vector(1024),
        metadata       JSONB NOT NULL DEFAULT '{}',
        created_at     TIMESTAMPTZ DEFAULT now()
    );
""")

cur.execute("""
    CREATE INDEX idx_resume_embedding
    ON chunks_resume
    USING hnsw (embedding vector_cosine_ops)
    WITH (m=16, ef_construction=64);
""")

cur.execute("""
    CREATE INDEX idx_resume_metadata
    ON chunks_resume USING gin(metadata);
""")

conn.commit()
print("Tables created: documents_resume, chunks_resume")
print("Indexes created: HNSW (embedding), GIN (metadata)")

Tables created: documents_resume, chunks_resume
Indexes created: HNSW (embedding), GIN (metadata)


## **Cell 4 — parse resume PDF with Docling**
- **Why Docling over PyPDF2:** Your resume has structured sections — Education, Experience, Projects, Skills. Docling preserves these as markdown headers which become section metadata on every chunk. PyPDF2 would flatten them into one block of text.
- **Path:**  data/docs/md_al_amin_resume.pdf — the file you uploaded.

In [4]:
PDF_PATH = Path("../data/docs/md_al_amin_resume.pdf")

if not PDF_PATH.exists():
    print(f"PDF not found at {PDF_PATH}")
    print("Make sure md_al_amin_resume.pdf is in data/docs/")
else:
    print(f"Found: {PDF_PATH}")
    print(f"Size : {PDF_PATH.stat().st_size / 1024:.1f} KB")

converter = DocumentConverter()
print("\nParsing with Docling...")
result   = converter.convert(str(PDF_PATH))
markdown = result.document.export_to_markdown()

print(f"Parsed successfully")
print(f"Characters: {len(markdown)}")
print(f"\nMarkdown preview:")
print(markdown[:600])

Found: ../data/docs/md_al_amin_resume.pdf
Size : 117.2 KB

Parsing with Docling...


[INFO] 2026-05-01 10:33:06,496 [RapidOCR] base.py:22: Using engine_name: onnxruntime
[INFO] 2026-05-01 10:33:06,502 [RapidOCR] download_file.py:60: File exists and is valid: /home/md-al-amin/My-Projects/End-to-End-Agentic-Ai-Automation-Lab/.venv/lib/python3.12/site-packages/rapidocr/models/ch_PP-OCRv4_det_infer.onnx
[INFO] 2026-05-01 10:33:06,503 [RapidOCR] main.py:53: Using /home/md-al-amin/My-Projects/End-to-End-Agentic-Ai-Automation-Lab/.venv/lib/python3.12/site-packages/rapidocr/models/ch_PP-OCRv4_det_infer.onnx
[INFO] 2026-05-01 10:33:06,565 [RapidOCR] base.py:22: Using engine_name: onnxruntime
[INFO] 2026-05-01 10:33:06,566 [RapidOCR] download_file.py:60: File exists and is valid: /home/md-al-amin/My-Projects/End-to-End-Agentic-Ai-Automation-Lab/.venv/lib/python3.12/site-packages/rapidocr/models/ch_ppocr_mobile_v2.0_cls_infer.onnx
[INFO] 2026-05-01 10:33:06,567 [RapidOCR] main.py:53: Using /home/md-al-amin/My-Projects/End-to-End-Agentic-Ai-Automation-Lab/.venv/lib/python3.12/site

Parsed successfully
Characters: 3766

Markdown preview:
## Education

## North South University

Bachelor of Science in Computer Science and Engineering

Jan 18 - Apr 25

CGPA: 3.17 (86%)

|

Coursework: Data Structures, Algorithms, DBMS, ML, DL, NLP, Operating Systems

## Technical Skills

Languages : Python | C/C++ | SQL | JavaScript

GenAI &amp; ML : PyTorch | TensorFlow | LLM | LangChain | LangGraph | vLLM | MCP | Prompt and Context Engineering | Embedding | RAG | Multi-Model RAG | PEFT(LoRA, QLoRA) | VectorDB | OpenAI API | Whisper | STT Web &amp; DevOps : FastAPI, Docker, GitLab CI/CD, Redis, NGiNX, PostgreSQL, RunPod, Playwright, Microservic


## **Cell 5 — chunking functions**
- **Why smaller chunks for a resume:** A resume has dense, structured content — each bullet point is a complete fact. We use child_size=2 sentences and parent_size=4 sentences. Larger windows would blend Skills with Experience which hurts retrieval precision.
- **Section detection:** The detect_section function walks backwards through the markdown lines to find the nearest ## header. Every chunk gets tagged with its section — "Experience", "Projects", "Skills" etc. This enables filtered queries like "only search within Skills section".

In [5]:
def detect_section(lines: list[str], up_to: int) -> str:
    """Walk backwards from current position to find nearest ## header."""
    for line in reversed(lines[:up_to]):
        line = line.strip()
        if line.startswith("## "):
            return line.lstrip("#").strip()
        if line.startswith("# "):
            return line.lstrip("#").strip()
    return "General"

def split_sentences(text: str) -> list[str]:
    """Split into sentences, skip headers and short fragments."""
    sentences = []
    for line in text.split("\n"):
        line = line.strip()
        if not line or line.startswith("#"):
            continue
        parts = re.split(r'(?<=[.!?])\s+', line)
        for part in parts:
            if len(part.strip()) > 15:
                sentences.append(part.strip())
    return sentences

def build_chunks(markdown: str, filename: str,
                 child_size: int = 2,
                 parent_size: int = 4,
                 overlap: int = 1) -> list[dict]:
    """
    Build parent-child chunks from markdown text.

    child_size=2  — 2 sentences per child (precise retrieval embedding)
    parent_size=4 — 4 sentences per parent (full context for LLM)
    overlap=1     — 1 sentence shared between adjacent chunks
                    prevents answers that span chunk boundaries from being missed

    Returns list of dicts with content, parent_content, section, metadata.
    """
    sentences = split_sentences(markdown)
    lines     = markdown.split("\n")
    chunks    = []
    idx       = 0
    i         = 0

    while i < len(sentences):
        parent_sents   = sentences[i : i + parent_size]
        parent_content = " ".join(parent_sents)
        section        = detect_section(lines, min(i * 2, len(lines) - 1))

        j = 0
        while j < len(parent_sents):
            child_content = " ".join(parent_sents[j : j + child_size])
            if len(child_content.strip()) > 15:
                chunks.append({
                    "content"       : child_content,
                    "parent_content": parent_content,
                    "section"       : section,
                    "metadata"      : {
                        "filename"   : filename,
                        "section"    : section,
                        "chunk_index": idx,
                    }
                })
                idx += 1
            j += child_size - overlap
        i += parent_size - overlap

    return chunks

chunks = build_chunks(markdown, "md_al_amin_resume.pdf")
print(f"Total chunks: {len(chunks)}")
print(f"\nSection distribution:")
for section, count in Counter(c["section"] for c in chunks).most_common():
    print(f"  {section:<35} {count:>3} chunks")
print(f"\nSample chunk:")
print(f"  Child  : {chunks[0]['content'][:100]}")
print(f"  Parent : {chunks[0]['parent_content'][:100]}")

Total chunks: 41

Section distribution:
  North South University                8 chunks
  Genuine Technology &amp; Research Ltd   8 chunks
  Jr. Generative AI Engineer            8 chunks
  1. SynapseAI High-Performance Memory Agent | FastAPI, LangGraph, Redis, PostgreSQL   8 chunks
  General                               4 chunks
  Technical Skills                      4 chunks
  3. Agentic AI Math Tutor with Adaptive Prompting | LangGraph, ChromaDB, Llama-4 [Capstone Proj-Apr 2025]   1 chunks

Sample chunk:
  Child  : Bachelor of Science in Computer Science and Engineering CGPA: 3.17 (86%)
  Parent : Bachelor of Science in Computer Science and Engineering CGPA: 3.17 (86%) Coursework: Data Structures
